In [ ]:
!apt-get install -y -q tesseract-ocr
!pip install -q pytesseract opencv-python-headless

import re
import cv2
import pytesseract
import matplotlib.pyplot as plt
from google.colab import files

up = files.upload()
path = list(up.keys())[0]

img = cv2.imread(path)
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
gray = cv2.resize(gray, None, fx=2, fy=2, interpolation=cv2.INTER_CUBIC)
gray = cv2.GaussianBlur(gray, (3, 3), 0)
thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)[1]

plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.show()

text = pytesseract.image_to_string(thresh)
print(text)

lines = [l.strip() for l in text.split("\n") if l.strip()]
name = lines[0] if lines else "not found"
dose = re.findall(r"\d+(?:\.\d+)?\s?(?:mg|mcg|g|ml|iu)\b", text, re.I)
expiry = re.findall(r"(?:exp(?:iry)?(?:\.|ires)?\s*(?:date)?\s*[:\-]?\s*)([A-Za-z0-9/\-\. ]{4,12})", text, re.I)
mfg = re.findall(r"(?:mfg|manufactured|mfd)\.?\s*(?:date)?\s*[:\-]?\s*([A-Za-z0-9/\-\. ]{4,12})", text, re.I)
batch = re.findall(r"(?:batch|b\.?\s?no|lot)\.?\s*(?:no)?\.?\s*[:\-]?\s*([A-Za-z0-9\-]{3,})", text, re.I)
warn = [l for l in lines if re.search(r"warning|keep out|do not|store|consult|caution", l, re.I)]
use = [l for l in lines if re.search(r"tablet|capsule|syrup|take|dose|daily|twice|once", l, re.I)]

print("medicine name:", name)
print("strength:", dose if dose else "not found")
print("expiry:", expiry[0].strip() if expiry else "not found")
print("manufactured:", mfg[0].strip() if mfg else "not found")
print("batch:", batch[0] if batch else "not found")
print("usage:", use if use else "not found")
print("warnings:", warn if warn else "not found")

reply = "this is " + name
if dose:
    reply += ", strength " + dose[0]
if expiry:
    reply += ", expires " + expiry[0].strip()
if warn:
    reply += ". please note: " + warn[0]
print(reply)

In [ ]:
!pip install -q deepface tf-keras

import cv2
import matplotlib.pyplot as plt
from google.colab import files
from deepface import DeepFace

up = files.upload()
path = list(up.keys())[0]

res = DeepFace.analyze(img_path=path, actions=["emotion"], enforce_detection=False)
res = res if isinstance(res, list) else [res]

img = cv2.imread(path)
replies = {
    "happy": "you look happy today, glad to see that. is there anything i can help you with",
    "sad": "you seem a little sad. i am here if you want to talk or listen to some music",
    "angry": "you look upset. take a deep breath, and tell me what went wrong so i can help",
    "fear": "you seem worried. everything will be fine, let me help you step by step",
    "surprise": "you look surprised. did something unexpected happen",
    "disgust": "something seems to be bothering you. shall i change what we are doing",
    "neutral": "you look calm. how can i help you today"
}

for r in res:
    emo = r["dominant_emotion"]
    box = r["region"]
    x, y, w, h = box["x"], box["y"], box["w"], box["h"]
    cv2.rectangle(img, (x, y), (x + w, y + h), (0, 255, 0), 2)
    cv2.putText(img, emo, (x, max(y - 10, 20)), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)
    print(emo)
    print({k: round(v, 2) for k, v in r["emotion"].items()})
    print(replies[emo])

plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.show()